# Chroma CRUD Operations

This notebook walks through create, read, update, and delete operations with a local Chroma vector store.

In [5]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

## 1. Set Up Paths and the Vector Store

In [6]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('d:/rag-vector-stores')

In [7]:
# Load environment variables from the local .env file.
dotenv_path = project_root / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: d:\rag-vector-stores\.env


In [24]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo_2"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: d:\rag-vector-stores\notebooks\chroma_langchain_db


In [ ]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

In [26]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


## 2. Add Small Helper Functions

In [11]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

## 3. Create and Insert Example Documents

In [12]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [14]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [19]:
print(uuid4())

77f328dd-783d-4fa4-9d5e-bd9cfd2cb1b5


In [27]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=73a1c0e2-e6d2-40c2-a128-6d244087f245
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=f8efa688-bc5a-4a32-a2e6-c7d1806b0a28
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=ff63b73b-3685-4373-840b-ce0ed60b448c
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=c07ea090-cbcb-45cf-a155-32069b5c40d1
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=3804cd92-b364-411d-a662-9cfc53650cd4
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [28]:
documents[0].id

'73a1c0e2-e6d2-40c2-a128-6d244087f245'

In [29]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
73a1c0e2-e6d2-40c2-a128-6d244087f245
f8efa688-bc5a-4a32-a2e6-c7d1806b0a28
ff63b73b-3685-4373-840b-ce0ed60b448c
c07ea090-cbcb-45cf-a155-32069b5c40d1
65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504
3804cd92-b364-411d-a662-9cfc53650cd4
c922f112-7840-4b55-989f-c34f6e160627
b7e70228-979f-4147-b367-6451a5a0f449
8ec0cbd3-1bbe-42a2-a896-e3acf89a287d
81cffbe4-74cb-408f-bc8a-5c32f1c12ccc

Total inserted documents: 10


## 4. Read the Stored Data Back

In [33]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [42]:
raw_records

{'ids': ['73a1c0e2-e6d2-40c2-a128-6d244087f245',
  'f8efa688-bc5a-4a32-a2e6-c7d1806b0a28',
  'ff63b73b-3685-4373-840b-ce0ed60b448c',
  'c07ea090-cbcb-45cf-a155-32069b5c40d1',
  '65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504',
  '3804cd92-b364-411d-a662-9cfc53650cd4',
  'c922f112-7840-4b55-989f-c34f6e160627',
  'b7e70228-979f-4147-b367-6451a5a0f449',
  '8ec0cbd3-1bbe-42a2-a896-e3acf89a287d',
  '81cffbe4-74cb-408f-bc8a-5c32f1c12ccc'],
 'embeddings': array([[ 0.00488281,  0.02095032,  0.01655579, ...,  0.00069189,
         -0.01637268,  0.02790833],
        [-0.01448822, -0.00543976,  0.03039551, ..., -0.02853394,
         -0.00193405,  0.04272461],
        [ 0.0226593 ,  0.01534271,  0.04550171, ...,  0.02297974,
          0.00804901, -0.01585388],
        ...,
        [ 0.00803375,  0.02313232,  0.02172852, ..., -0.02070618,
         -0.01786804,  0.01334381],
        [ 0.00882721,  0.06274414,  0.09967041, ..., -0.01580811,
         -0.01374817,  0.03677368],
        [ 0.01803589,  0.08557129, 

In [39]:
print(raw_records["embeddings"][0:2, 0:20].shape)

(2, 20)


In [40]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
73a1c0e2-e6d2-40c2-a128-6d244087f245
f8efa688-bc5a-4a32-a2e6-c7d1806b0a28
ff63b73b-3685-4373-840b-ce0ed60b448c


In [45]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['b7e70228-979f-4147-b367-6451a5a0f449',
 '8ec0cbd3-1bbe-42a2-a896-e3acf89a287d',
 '81cffbe4-74cb-408f-bc8a-5c32f1c12ccc']

In [46]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=b7e70228-979f-4147-b367-6451a5a0f449
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=8ec0cbd3-1bbe-42a2-a896-e3acf89a287d
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=81cffbe4-74cb-408f-bc8a-5c32f1c12ccc
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



In [47]:
print(selected_documents)

[Document(id='b7e70228-979f-4147-b367-6451a5a0f449', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'), Document(id='8ec0cbd3-1bbe-42a2-a896-e3acf89a287d', metadata={'doc_number': 9, 'topic': 'Cricket'}, page_content='Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.'), Document(id='81cffbe4-74cb-408f-bc8a-5c32f1c12ccc', metadata={'doc_number': 10, 'topic': 'Cricket'}, page_content='A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.')]


## 5. Run a Similarity Search

In [48]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [49]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=c07ea090-cbcb-45cf-a155-32069b5c40d1
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=b7e70228-979f-4147-b367-6451a5a0f449
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [50]:
search_results

[Document(id='c07ea090-cbcb-45cf-a155-32069b5c40d1', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='b7e70228-979f-4147-b367-6451a5a0f449', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [52]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='c07ea090-cbcb-45cf-a155-32069b5c40d1', metadata={'doc_number': 4, 'topic': 'RAG'}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.8067810535430908),
 (Document(id='b7e70228-979f-4147-b367-6451a5a0f449', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.9389221668243408),
 (Document(id='65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.0750246047973633),
 (Document(id='c922f112-7840-4b55-989f-c34f6e160627', metadata={'topic': 'LLM', 'doc_number': 7}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  1.130632996559143)]

## 6. Update Existing Documents

In [53]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['c07ea090-cbcb-45cf-a155-32069b5c40d1',
 'b7e70228-979f-4147-b367-6451a5a0f449']

In [54]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=c07ea090-cbcb-45cf-a155-32069b5c40d1
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=b7e70228-979f-4147-b367-6451a5a0f449
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [60]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [55]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
c07ea090-cbcb-45cf-a155-32069b5c40d1
b7e70228-979f-4147-b367-6451a5a0f449


In [56]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=c07ea090-cbcb-45cf-a155-32069b5c40d1
metadata={'doc_number': 4, 'topic': 'RAG'}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=b7e70228-979f-4147-b367-6451a5a0f449
metadata={'topic': 'LLM', 'doc_number': 8}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [57]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [59]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=c07ea090-cbcb-45cf-a155-32069b5c40d1
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [61]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['8ec0cbd3-1bbe-42a2-a896-e3acf89a287d',
 '81cffbe4-74cb-408f-bc8a-5c32f1c12ccc']

In [62]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
8ec0cbd3-1bbe-42a2-a896-e3acf89a287d
81cffbe4-74cb-408f-bc8a-5c32f1c12ccc


In [64]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
73a1c0e2-e6d2-40c2-a128-6d244087f245
f8efa688-bc5a-4a32-a2e6-c7d1806b0a28
ff63b73b-3685-4373-840b-ce0ed60b448c
c07ea090-cbcb-45cf-a155-32069b5c40d1
65baf5b7-0aaf-4159-9cd5-cd9fc5c3e504
3804cd92-b364-411d-a662-9cfc53650cd4
c922f112-7840-4b55-989f-c34f6e160627
b7e70228-979f-4147-b367-6451a5a0f449

Deleted ids still present?
8ec0cbd3-1bbe-42a2-a896-e3acf89a287d: False
81cffbe4-74cb-408f-bc8a-5c32f1c12ccc: False


In [65]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
